# Coursework 1 Group (text)


Group number: 4

Student names and k-numbers:
1. Henry Longe - k2552174
2. Nsetu Tarimo - k2604076
3. Thaw Zin Lin Myat - k2543493
4. Henry Akwarandu - k2557070

# Load modules (code)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.figure_factory as ff
from sklearn.model_selection import train_test_split

#sklearn classification models 
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier


#Pipeline
from sklearn.pipeline import Pipeline

#preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import label_binarize

#Dimensionality Reduction
from sklearn.decomposition import PCA

#Hyperparameter Tuning
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

#Evaluation Metrics
from sklearn.metrics import balanced_accuracy_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve

#Confusion Matrix
from sklearn.metrics import confusion_matrix

#OVR
from sklearn.multiclass import OneVsRestClassifier



# Load data (code)

In [2]:
#data load
from sklearn.datasets import fetch_openml

dataset = fetch_openml(data_id=4538, as_frame=False) # fetch the dataset

In [3]:
# X and y
X = dataset.data
y = dataset.target


print(X.shape)
print(y.shape)

#checking integrity of data
distribution = np.unique(y, return_counts=True)
isnan = np.isnan(X).sum()

print("general distribution of data :", distribution)
print("Nans in data :", isnan)


(9873, 32)
(9873,)
general distribution of data : (array(['D', 'H', 'P', 'R', 'S'], dtype=object), array([2741,  998, 2097, 1087, 2950]))
Nans in data : 0


In [4]:
from sklearn.ensemble import IsolationForest

iso = IsolationForest(
    contamination=0.05,
    random_state=42
)

outlier_labels = iso.fit_predict(X)
# -1 = outlier, 1 = inlier

n_outliers = (outlier_labels == -1).sum()
print("Detected outliers:", n_outliers)

Detected outliers: 494


In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, test_size=0.3, stratify=y, random_state=42)
#for KNN might delete later
le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)


# Classification

## Classification methods used (text)

Mention the classification methods used below. You should not describe them, but if they have not been discussed in the class, you should cite the source:


1.  Logistic Regression 
2.  
3. Random Forest without PCA
4. Gradient Boost with PCA 
5.
6.
7.
8.



## Training (code)

### Tuning Hyperparameters

In [ ]:
#hyperparameter tuning with and finding out the best hyper parameter
models_tuning = {
    "Logistic Regression": (
        Pipeline([
            ("scaler", RobustScaler()),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "clf__C": [0.001, 0.01, 0.1, 1, 10,100]
        }
    ),
    "Logistic Regression (PCA)": (
        Pipeline([
            ("scaler", RobustScaler()),
            ("pca", PCA()),
            ("clf", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "pca__n_components": [5 ,10, 15, 20],
            "clf__C": [0.001, 0.01, 0.1, 1, 10,100]
        }
    ),

    "Decision Tree": (
        Pipeline([
            ("dt", DecisionTreeClassifier(
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "dt__max_depth": [None ,5 , 10, 20],
            "dt__min_samples_leaf": [1, 5, 10]
        }
        
    ),
    "Decision Tree (PCA)": (
        Pipeline([
            ("pca", PCA()),
            ("dt", DecisionTreeClassifier(
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "pca__n_components": [5 ,10 ,15 ,20],
            "dt__max_depth": [None ,5 , 10, 20],
            "dt__min_samples_leaf": [1, 5, 10]
        }
        
    ),

    "Random Forest": (
        Pipeline([
            ("rf", RandomForestClassifier(
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "rf__n_estimators": [500],
            "rf__max_depth": [None],
            "rf__min_samples_leaf": [5]
        }
        
    ),
    "Random Forest (PCA)": (
        Pipeline([
            ("pca", PCA()),
            ("rf", RandomForestClassifier(
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "pca__n_components": [5 ,10 ,15 ,20],
            "rf__n_estimators": [200],
            "rf__max_depth": [None],
            "rf__min_samples_leaf": [5]
        }
    ),  

    "KNN": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(
                algorithm="brute",
            ))
        ]),
        {
            "clf__n_neighbors": list(range(1, 51, 2)),
            "clf__weights": ["uniform", "distance"],
            "clf__metric": ["euclidean", "manhattan"]
        }
    ),
    "KNN (PCA)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA()),
            ("clf", KNeighborsClassifier(
                algorithm="brute",
            ))
        ]),
        {
            "pca__n_components": [5 ,10 ,15 ,20],
            "clf__n_neighbors": list(range(1, 51, 2)),
            "clf__weights": ["uniform", "distance"],
            "clf__metric": ["euclidean", "manhattan"]
        }
    ),

    "Linear SVC": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LinearSVC(
                class_weight="balanced",
                max_iter=5000
            ))
        ]),
        {
            "clf__C": [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100], 
            "clf__loss": ["squared_hinge"],
            "clf__dual": [False] 
        }
    ),
    "Linear SVC (PCA)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA()),
            ("clf", LinearSVC(
                class_weight="balanced",
                max_iter=5000
            ))
        ]),
        {
            "pca__n_components": [5 ,10 ,15 ,20],
            "clf__C": [1e-4, 1e-3, 1e-2, 1e-1, 1, 10, 100], 
            "clf__loss": ["squared_hinge"],
            "clf__dual": [False] 
        }
    ),
    "SVC (RBF)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=False
            ))
        ]),
        {
            "clf__C": [0.1, 1, 10, 100],
            "clf__gamma": [0.001, 0.01, 0.1]
        }
    ),
    "SVC [RBF] (PCA)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA()),
            ("clf", SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=False
            ))
        ]),
        {
            "pca__n_components": [5 ,10 ,15 ,20],
            "clf__C": [0.1, 1, 10, 100],
            "clf__gamma": [0.001, 0.01, 0.1]
        }
    ),

    "Naive Bayes": (
        Pipeline([
            ("nb", GaussianNB())
        ]),
        {
            "nb__var_smoothing": np.logspace(-12, -6, 20)
        }
    ),
    "Naive Bayes (PCA)": (
        Pipeline([
            ("pca", PCA()),
            ("nb", GaussianNB())
        ]),
        {
            "pca__n_components": [5 ,10 ,15 ,20],
            "nb__var_smoothing": np.logspace(-12, -6, 20)

        }
    ),
    
    "Gradient Boost": (
        Pipeline([
            ("gb", GradientBoostingClassifier(random_state=42))
        ]),
        {
            "gb__n_estimators": [200,400],
            "gb__learning_rate": [0.01,0.1,1],
            "gb__max_depth": [5,10,15],
            "gb__subsample": [0.8,1.0]
        }
    ),
     "Gradient Boost (PCA)": (
        Pipeline([
            ("pca", PCA()),
            ("gb", GradientBoostingClassifier(random_state=42))
        ]),
        {
            "pca__n_components": [5,10,15,20],
            "gb__n_estimators": [200,400],
            "gb__learning_rate": [0.01,0.1,1],
            "gb__max_depth": [5,10,15],
            "gb__subsample": [0.8,1.0]
        }
    ),
}


tuned_results = {}
best_models = {}
best_params ={}

acc=0.0;
for name, (model, params) in models_tuning.items():
    print(f"\nTuning {name} ...")

    grid = GridSearchCV(
        model,
        params,
        scoring="balanced_accuracy",
        cv=5,
        n_jobs=-1
    )
    if name=="KNN":
        grid.fit(X_train, y_train_enc)    
    
        best_model = grid.best_estimator_
        y_pred_enc = best_model.predict(X_test)
        y_pred = le.inverse_transform(y_pred_enc)

    else:
        grid.fit(X_train, y_train)
    
        best_model = grid.best_estimator_
        y_pred = best_model.predict(X_test)
    
    
    acc = balanced_accuracy_score(y_test, y_pred)
    
    tuned_results[name] = acc
    best_models[name] = best_model

print("\nTuned Results (Balanced Accuracy):")
for k, v in tuned_results.items():
    print(f"{k}: {v:.4f}")

print("\nBest Model:")
for k, v in best_models.items():
    print(f"{k}: {v}")

print("\nBest Hyperparameter:")
for k, v in best_params.items():
    print(f"{k}: {v}")

### Hypertuned models

In [ ]:
models = {
    #  Logistic Regression 
    "Logistic Regression": (
        Pipeline([
            ("scaler", RobustScaler()),
            ("lr", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "lr__C":0.1,
        }
    ),

    "Logistic Regression (PCA)": (
        Pipeline([
            ("scaler", RobustScaler()),
            ("pca", PCA()),
            ("lr", LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "lr__C":0.001,
            "pca__n_components": 20,
        }
    ),

    #  Decision Tree 
    "Decision Tree": (
        Pipeline([
            ("dt", DecisionTreeClassifier(
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "dt__max_depth": 10,
            "dt__min_samples_leaf": 5,
        }
    ),

    "Decision Tree (PCA)": (
        Pipeline([
            ("pca", PCA()),
            ("dt", DecisionTreeClassifier(
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "dt__max_depth": 10,
            "dt__min_samples_leaf": 1,
            "pca__n_components": 10,
        }
    ),

    #  Random Forest 
    "Random Forest": (
        Pipeline([
            ("rf", RandomForestClassifier(
                n_estimators=500,
                max_depth=None,
                min_samples_leaf=5,
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "rf__n_estimators":500,
            "rf__max_depth": None,
            "rf__min_samples_leaf": 5,
        }
    ),

    "Random Forest (PCA)": (
        Pipeline([
            ("pca", PCA()),
            ("rf", RandomForestClassifier(
                class_weight="balanced",
                random_state=42
            ))
        ]),
        {
            "rf__n_estimators":200,
            "rf__max_depth": None,
            "rf__min_samples_leaf": 5,
            "pca__n_components": 20,
        }
    ),

    #  KNN 
    "KNN": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("knn", KNeighborsClassifier(
                weights="uniform",
                algorithm="brute",             
            ))
        ]),
        {
            "knn__n_neighbors": 1,
            "knn__metric": "manhattan",
        }
    ),

    "KNN (PCA)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA()),
            ("knn", KNeighborsClassifier(
                weights="uniform",
                algorithm="brute",
            ))
        ]),
        {
            "knn__n_neighbors": 1,
            "knn__metric": "euclidean",
            "pca__n_components": 20,
        }
    ),

    #  Linear SVC 
    "Linear SVC": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("lsvm", LinearSVC(
                loss="squared_hinge",
                dual=False,
                class_weight="balanced",
                max_iter=5000,
                random_state=42
            ))
        ]),
        {
            "lsvm__C": 10,
            "lsvm__loss": "squared_hinge",
        }
    ),

    "Linear SVC (PCA)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA()),
            ("lsvm", LinearSVC(
                dual=False,
                class_weight="balanced",
                max_iter=5000,
                random_state=42
            ))
        ]),
        {
            "lsvm__C": 1,
            "lsvm__loss": "squared_hinge",
            "pca__n_components": 20,
        }
    ),

    #  SVC RBF 
    "SVC (RBF)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("rbf", SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=False,
                random_state=42
            ))
        ]),
        {
            "rbf__C": 10,
            "rbf__gamma": 0.1,  
        }
    ),

    "SVC (RBF) (PCA)": (
        Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA()),
            ("rbf", SVC(
                kernel="rbf",
                class_weight="balanced",
                probability=False,           
                random_state=42

            ))
        ]),
        {
            "rbf__C": 10,
            "rbf__gamma": 0.1,  
            "pca__n_components": 15,
        }
    ),

    #  Naive Bayes 
    "Naive Bayes": (
        Pipeline([
            ("nb", GaussianNB())
        ]),
        {
            "nb__var_smoothing":1e-12,
        }
    ),

    "Naive Bayes (PCA)": (
        Pipeline([
            ("pca", PCA()),
            ("nb", GaussianNB())
        ]),
        {
            "nb__var_smoothing":1e-12,
            "pca__n_components": 15,
        }
    ),

    #  Gradient Boost
    "Gradient Boost": (
        Pipeline([
            ("gb", GradientBoostingClassifier(
                random_state=42
            ))
        ]),
        {
            "gb__n_estimators": 200,
            "gb__learning_rate": 0.1,
            "gb__max_depth": 5,
            "gb__subsample":0.8
        }
    ),

    "Gradient Boost (PCA)": (
        Pipeline([
            ("pca", PCA()),
            ("gb", GradientBoostingClassifier(
                random_state=42
            ))
        ]),
        {
            "gb__n_estimators": 200,
            "gb__learning_rate": 0.1,
            "gb__max_depth": 5,
            "gb__subsample":0.8,
            "pca__n_components": 20,
        }
    ),
}

## Evaluation (code)

### Evaluate function

In [7]:
def evaluate_models(models, X_train, X_test, y_train, y_test):

    classes = np.unique(y_train)
    n_classes = len(np.unique(y_train))
    y_test_bin = label_binarize(y_test, classes=np.unique(y_train))

    results = {}
    macro_rocs = {}
    micro_rocs = {}


    for name, (pipeline, params) in models.items():

        # Set best parameters
        if params:
            pipeline.set_params(**params)

        # Wrap in OneVsRest for ROC computation
        clf = OneVsRestClassifier(pipeline)
        clf.fit(X_train, y_train)

        y_pred = clf.predict(X_test)
      
        # Balanced Accuracy
        bal_acc = balanced_accuracy_score(y_test, y_pred)

        # Get probability
        if hasattr(clf, "predict_proba"):
            y_score = clf.predict_proba(X_test)
        else:
            y_score = clf.decision_function(X_test)

        # ROC AUC
        macro_auc = roc_auc_score(y_test_bin, y_score, average="macro")
        micro_auc = roc_auc_score(y_test_bin, y_score, average="micro")

        # #Just some Prints
        # print(f"Balanced Accuracy: {bal_acc:.4f}")
        # print(f"Macro-average OvR ROC AUC: {macro_auc:.4f}")
        # print(f"Micro-average OvR ROC AUC: {micro_auc:.4f}")

        # ---- Micro ROC ----
        fpr_micro, tpr_micro, _ = roc_curve(
            y_test_bin.ravel(), y_score.ravel()
        )

        per_class = {}
        fpr_dict = {}
        tpr_dict = {}

        for i in range(n_classes):
            fpr_i, tpr_i, _ = roc_curve(
                y_test_bin[:, i], y_score[:, i]
            )
            per_class[i] = (fpr_i, tpr_i)
            fpr_dict[i] = fpr_i
            tpr_dict[i] = tpr_i

        # ---- Macro ROC ----
        all_fpr = np.unique(
            np.concatenate([fpr_dict[i] for i in range(n_classes)])
        )

        mean_tpr = np.zeros_like(all_fpr)
        for i in range(n_classes):
            mean_tpr += np.interp(all_fpr, fpr_dict[i], tpr_dict[i])
        mean_tpr /= n_classes

        macro_rocs[name] = (all_fpr, mean_tpr)
        micro_rocs[name] = (fpr_micro,tpr_micro)

        #Confusion Matrix
        cm = confusion_matrix(y_test, y_pred, labels=classes)
        results[name] = {
            "balanced_accuracy": bal_acc,
            "macro_auc": macro_auc,
            "micro_auc": micro_auc,
            "confusion_matrix": cm,
            "roc": {
                "macro": macro_rocs[name],
                "micro": micro_rocs[name],
                "per_class": per_class
            }
        }

    # =============================
    # Summary Table
    # =============================
    print("\n" + "="*80)
    print("Model PERFORMANCE SUMMARY")
    print("="*80)

    for name, metrics in results.items():
        print(f"{name}")
        print(f"  Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
        print(f"  Macro-average ROC AUC: {metrics['macro_auc']:.4f}")
        print(f"  Micro-average ROC AUC: {metrics['micro_auc']:.4f}")
        print("-"*60)
    return results, classes

### Plot Functions

In [8]:
def plot_macro_roc(results):

    fig = go.Figure()

    for name, res in results.items():
        fpr, tpr = res["roc"]["macro"]
        auc = res["macro_auc"]

        fig.add_trace(go.Scatter(
            x=fpr,
            y=tpr,
            mode="lines",
            name=f"{name} (AUC={auc:.3f})"
        ))

    fig.add_trace(go.Scatter(
        x=[0, 1], 
        y=[0, 1],
        mode="lines",
        line=dict(dash="dash"),
        name="Random"
    ))

    fig.update_layout(
        title="Macro-average One-vs-Rest ROC Curves",
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate"
    )

    fig.show()

In [9]:
def plot_micro_roc(results):

    fig = go.Figure()

    for name, res in results.items():
        fpr, tpr = res["roc"]["micro"]
        auc = res["micro_auc"]

        fig.add_trace(go.Scatter(
            x=fpr,
            y=tpr,
            mode="lines",
            name=f"{name} (AUC={auc:.3f})"
        ))

    fig.add_trace(go.Scatter(
        x=[0, 1], y=[0, 1],
        mode="lines",
        line=dict(dash="dash"),
        name="Random"
    ))

    fig.update_layout(
        title="Micro-average One-vs-Rest ROC Curves",
        xaxis_title="False Positive Rate",
        yaxis_title="True Positive Rate"
    )

    fig.show()

In [10]:
def plot_per_class_roc(results, classes):

    for i, cls in enumerate(classes):
        fig = go.Figure()

        for name, res in results.items():
            fpr, tpr = res["roc"]["per_class"][i]
            fig.add_trace(go.Scatter(
                x=fpr,
                y=tpr,
                mode="lines",
                name=name
            ))

        fig.add_trace(go.Scatter(
            x=[0, 1], y=[0, 1],
            mode="lines",
            line=dict(dash="dash"),
            name="Random"
        ))

        fig.update_layout(
            title=f"One-vs-Rest ROC Curves – Class {cls}",
            xaxis_title="False Positive Rate",
            yaxis_title="True Positive Rate"
        )

        fig.show()

In [11]:
def plot_confusion_matrices(results, classes):

    for name, res in results.items():
        cm = res["confusion_matrix"]

        fig = ff.create_annotated_heatmap(
            z=cm,
            x=[f"Pred {c}" for c in classes],
            y=[f"True {c}" for c in classes],
            colorscale="Blues"
        )

        fig.update_layout(
            title=f"Confusion Matrix – {name}",
            xaxis=dict(title="Predicted Label", side="bottom"),
            yaxis=dict(title="True Label")
        )

        fig.show()

### Results

In [12]:
results, classes = evaluate_models(
    models, X_train, X_test, y_train, y_test
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/joblib/externals/loky/backend/context.py:131: UserWarning:

Could not find the number of physical cores for the following reason:
[Errno 2] No such file or directory: 'sysctl'
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.

  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/joblib/externals/loky/backend/context.py", line 249, in _count_physical_cores
    cpu_count_physical = _count_physical_cores_darwin()
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/joblib/externals/loky/backend/context.py", line 312, in _count_physical_cores_darwin
    cpu_info = subprocess.run(
        "sysctl -n hw.physicalcpu".split(),
        capture_output=True,
        text=True,
    )
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13


Model PERFORMANCE SUMMARY
Logistic Regression
  Balanced Accuracy: 0.4390
  Macro-average ROC AUC: 0.7533
  Micro-average ROC AUC: 0.7576
------------------------------------------------------------
Logistic Regression (PCA)
  Balanced Accuracy: 0.3978
  Macro-average ROC AUC: 0.7126
  Micro-average ROC AUC: 0.7159
------------------------------------------------------------
Decision Tree
  Balanced Accuracy: 0.4824
  Macro-average ROC AUC: 0.7406
  Micro-average ROC AUC: 0.7655
------------------------------------------------------------
Decision Tree (PCA)
  Balanced Accuracy: 0.4644
  Macro-average ROC AUC: 0.7359
  Micro-average ROC AUC: 0.7547
------------------------------------------------------------
Random Forest
  Balanced Accuracy: 0.5893
  Macro-average ROC AUC: 0.8800
  Micro-average ROC AUC: 0.8932
------------------------------------------------------------
Random Forest (PCA)
  Balanced Accuracy: 0.5602
  Macro-average ROC AUC: 0.8623
  Micro-average ROC AUC: 0.8781
--

### Plot Results

In [13]:
plot_macro_roc(results)
plot_micro_roc(results)

In [14]:
plot_per_class_roc(results, classes)

In [15]:
plot_confusion_matrices(results, classes)

# References (text)

List any references you may have used in your document before, using one of the established referencing system (e.g. IEEE, Harvard, etc).